# HYG/LQD 分红／拆股原始事件补充

固定两条 query2 URL，仅将 `events` 改为 `div,splits`，原始收盘数组必须逐值等于先前已归档响应；保留事件原字节及 NAS 回读。此阶段不计算总收益或预测。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,math
from pathlib import Path
from datetime import datetime,timezone
from zoneinfo import ZoneInfo
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_yahoo_credit_etf_history_source_20261023';plan=json.loads((card/'event_supplement_plan.json').read_text());parent=json.loads((card/'source_result.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'source_result.json')==plan['parent_source_result_sha256'] and parent['status']=='HYG_LQD_current_chart_raw_full_span_source_received_PIT_pending' and not (card/'event_attempt_started.json').exists()
raw=root/'data/snapshots/stage1_yahoo_credit_etf_history_source_20261023/events';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'event_attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests']):
 state['GET_attempts']+=1;(card/'event_attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 symbol=spec['symbol'];body=raw/(symbol+'.json');headers=raw/(symbol+'.headers');started=datetime.now(timezone.utc).isoformat();r=plan['request']
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'symbol':symbol,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'symbol':symbol,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>r['max_response_bytes_each']:failure='transport_or_size_failure';break
 try:
  payload=json.loads(body.read_text());chart=payload['chart'];assert chart['error'] is None;z=chart['result'];assert len(z)==1 and z[0]['meta']['symbol']==symbol
  z=z[0];dates=[datetime.fromtimestamp(t,timezone.utc).astimezone(ZoneInfo('America/New_York')).date() for t in z['timestamp']];close=z['indicators']['quote'][0]['close'];assert len(dates)==len(close)>3000 and len(dates)==len(set(dates))
  original=json.loads((root/'data/snapshots/stage1_yahoo_credit_etf_history_source_20261023'/(symbol+'.json')).read_text())['chart']['result'][0]
  assert z['timestamp']==original['timestamp'] and z['indicators']['quote'][0]['close']==original['indicators']['quote'][0]['close']
  events=z.get('events') or {};div=events.get('dividends') or {};splits=events.get('splits') or {};assert len(div)>100
  ddates=[datetime.fromtimestamp(int(key),timezone.utc).astimezone(ZoneInfo('America/New_York')).date() for key in div]
  assert len(ddates)==len(set(ddates)) and all(float(x['amount'])>0 and math.isfinite(float(x['amount'])) for x in div.values())
  checks.append({'symbol':symbol,'status':'original_dividend_event_list_received_source_only','raw_sha256':receipt['raw_sha256'],'raw_bytes':receipt['raw_bytes'],'price_rows_equal_parent':len(dates),'dividend_events':len(div),'first_dividend_date':str(min(ddates)),'last_dividend_date':str(max(ddates)),'split_events':len(splits),'current_retrospective_events_not_historical_vintage':True})
 except Exception as exc:failure='event_schema_or_price_identity_failure';checks.append({'symbol':symbol,'status':failure,'error':type(exc).__name__+':'+str(exc)[:150],'raw_sha256':receipt['raw_sha256']});break
 if i+1<len(plan['exact_requests']):time.sleep(r['seconds_between_requests'])
(card/'event_capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'event_supplement_plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'event_capture_receipts.json')};(card/'event_source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='hyg-lqd-events-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'event_source_manifest.json',arcname='event_source_manifest.json');out.add(card/'event_capture_receipts.json',arcname='event_capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='hyg-lqd-event-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'HYG_LQD_dividend_event_raw_sources_received_historical_vintage_pending' if failure is None and len(checks)==2 else 'bounded_credit_ETF_event_source_partial_or_failed','failure':failure,'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'event_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'bounded_credit_ETF_event_source_partial_or_failed', 'failure': 'event_schema_or_price_identity_failure', 'GETs': 1, 'checks': [{'symbol': 'HYG', 'status': 'event_schema_or_price_identity_failure', 'error': 'AssertionError:', 'raw_sha256': '6259b766f7b89c47af1ffebd1e6bdc1d3aabc39c81c81b8690be95e3b9a280f6'}], 'NAS_sha256': '0f62d6e462f095376abafacdd23d5a2cfac315d485f6b14cff348dad6d0d69f9'}


## 中文结论

事件虽被当前来源接收，历史首次公布金额及历史调整价版本仍需资格化；本卡 0 SOXX 收益、0 拟合、0 账户。